In [1]:
"""
P4 - Evasion de Colisiones (Problema Abierto)
MyCobot 280 - 6 DOF

Objetivo:
Implementar un sistema de prevención de colisiones
para el MyCobot 280 usando:

1. Límites conservadores por joint
2. Verificación de altura mínima mediante FK
3. Waypoint seguro de clearance

Autor: [Tu nombre]
Curso: Robótica
"""

'\nP4 - Evasion de Colisiones (Problema Abierto)\nMyCobot 280 - 6 DOF\n\nObjetivo:\nImplementar un sistema de prevención de colisiones\npara el MyCobot 280 usando:\n\n1. Límites conservadores por joint\n2. Verificación de altura mínima mediante FK\n3. Waypoint seguro de clearance\n\nAutor: [Tu nombre]\nCurso: Robótica\n'

In [2]:
import time
import logging
import numpy as np

from math import (
    cos,
    sin,
    radians
)

from pymycobot.mycobot import MyCobot

In [5]:
print("Conectando robot...")

mc = MyCobot('/dev/ttyUSB0', 1000000)

time.sleep(2)

print("Conexion:", mc.is_controller_connected())

mc.power_on()

time.sleep(2)

print("Robot conectado correctamente")

Conectando robot...
Conexion: 1
Robot conectado correctamente


In [6]:
COLLISION_SCENARIOS = {

    "colision_mesa": {

        "descripcion":
            "El brazo impacta la mesa.",

        "causa":
            "Altura Z demasiado baja.",

        "prevencion":
            "Verificacion de Z minima."
    },

    "auto_colision": {

        "descripcion":
            "El codo golpea otra parte del robot.",

        "causa":
            "J2 y J3 en angulos extremos.",

        "prevencion":
            "Limites conservadores."
    },

    "colision_objetos": {

        "descripcion":
            "Choque con contenedores o camara.",

        "causa":
            "Movimiento lateral peligroso.",

        "prevencion":
            "Waypoint seguro."
    }
}

for k, v in COLLISION_SCENARIOS.items():

    print(f"\n{k}")

    print(v)


colision_mesa
{'descripcion': 'El brazo impacta la mesa.', 'causa': 'Altura Z demasiado baja.', 'prevencion': 'Verificacion de Z minima.'}

auto_colision
{'descripcion': 'El codo golpea otra parte del robot.', 'causa': 'J2 y J3 en angulos extremos.', 'prevencion': 'Limites conservadores.'}

colision_objetos
{'descripcion': 'Choque con contenedores o camara.', 'causa': 'Movimiento lateral peligroso.', 'prevencion': 'Waypoint seguro.'}


In [7]:
CONSERVATIVE_JOINT_LIMITS = {

    0: (-160, 160),
    1: (-110, 70),
    2: (-120, 120),
    3: (-130, 130),
    4: (-150, 150),
    5: (-175, 175),
}

Z_MIN_SAFE = 60.0

SAFE_WAYPOINT_ANGLES = [
    0,
    0,
    -90,
    90,
    0,
    -45
]

In [8]:
def forward_kinematics(angles):

    theta1 = radians(angles[0])
    theta2 = radians(angles[1])
    theta3 = radians(angles[2])

    L1 = 131.0
    L2 = 110.0
    L3 = 96.0

    r = (
        L2 * cos(theta2) +
        L3 * cos(theta2 + theta3)
    )

    x = r * cos(theta1)

    y = r * sin(theta1)

    z = (
        L1 +
        L2 * sin(theta2) +
        L3 * sin(theta2 + theta3)
    )

    T = np.eye(4)

    T[0, 3] = x
    T[1, 3] = y
    T[2, 3] = z

    return T

In [9]:
def extract_position(T):

    x = T[0, 3]
    y = T[1, 3]
    z = T[2, 3]

    return x, y, z

In [10]:
class CollisionChecker:

    def __init__(self, mc=None):

        self.mc = mc

        self.joint_limits = \
            CONSERVATIVE_JOINT_LIMITS

        self.z_min = Z_MIN_SAFE


    # ==================================================
    # VERIFICAR LIMITES
    # ==================================================

    def check_joint_limits(self, angles):

        for i, angle in enumerate(angles):

            lo, hi = self.joint_limits[i]

            if not (lo <= angle <= hi):

                return False, \
                    f"J{i+1} fuera de limite"

        return True, "OK"


    # ==================================================
    # VERIFICAR ALTURA
    # ==================================================

    def check_min_height(self, angles):

        T = forward_kinematics(angles)

        x, y, z = extract_position(T)

        if z < self.z_min:

            return False, z, \
                f"Z={z:.1f} mm peligrosa"

        return True, z, \
            f"Z={z:.1f} mm segura"


    # ==================================================
    # MOVIMIENTO SEGURO
    # ==================================================

    def safe_move(self, target_angles, speed=20):

        valid_l, msg_l = \
            self.check_joint_limits(
                target_angles
            )

        if not valid_l:

            print(msg_l)

            return False

        valid_z, z_val, msg_z = \
            self.check_min_height(
                target_angles
            )

        if not valid_z:

            print(msg_z)

            return False

        print(msg_z)

        if self.mc is not None:

            print("Moviendo waypoint...")

            self.mc.send_angles(
                SAFE_WAYPOINT_ANGLES,
                speed
            )

            time.sleep(3)

            print("Moviendo destino...")

            self.mc.send_angles(
                target_angles,
                speed
            )

            time.sleep(4)

        return True

In [11]:
checker = CollisionChecker(mc=mc)

print("CollisionChecker listo")

CollisionChecker listo


In [14]:
angles_safe = [
    0,
    0,
    0,
    0,
    0,
    -45
]

checker.safe_move(angles_safe)

Z=131.0 mm segura
Moviendo waypoint...
Moviendo destino...


True

In [15]:
angles_danger = [
    0,
    -120,
    0,
    0,
    0,
    0
]

checker.safe_move(angles_danger)

J2 fuera de limite


False

In [19]:
test_sequences = [

    [0,0,0,0,0,-45],

    [30,0,0,-45,0,0],

    [0,-100,50,0,0,0],

    [0,-120,0,0,0,0]
]

for angles in test_sequences:

    print("\nProbando:", angles)

    checker.safe_move(angles)


Probando: [0, 0, 0, 0, 0, -45]
Z=131.0 mm segura
Moviendo waypoint...
Moviendo destino...

Probando: [30, 0, 0, -45, 0, 0]
Z=131.0 mm segura
Moviendo waypoint...
Moviendo destino...

Probando: [0, -100, 50, 0, 0, 0]
Z=-50.9 mm peligrosa

Probando: [0, -120, 0, 0, 0, 0]
J2 fuera de limite


In [17]:
coords = mc.get_coords()

print("\nCoordenadas reales:")

print(coords)


Coordenadas reales:
[190.3, 32.6, 283.6, -165.49, -0.13, -59.85]


In [18]:
"""
CONCLUSIONES

1. Se identificaron colisiones potenciales:
   - mesa
   - auto-colision
   - objetos externos

2. Se implementaron mecanismos preventivos:
   - limites conservadores
   - verificacion de altura
   - waypoint seguro

3. El robot logro ejecutar trayectorias
   seguras sin colisiones.

4. Los movimientos peligrosos fueron
   bloqueados correctamente.

5. Limitaciones:
   - FK simplificada
   - no se usan sensores externos
   - no existe planificacion dinamica

6. El sistema es adecuado para
   entornos controlados de laboratorio.
"""

'\nCONCLUSIONES\n\n1. Se identificaron colisiones potenciales:\n   - mesa\n   - auto-colision\n   - objetos externos\n\n2. Se implementaron mecanismos preventivos:\n   - limites conservadores\n   - verificacion de altura\n   - waypoint seguro\n\n3. El robot logro ejecutar trayectorias\n   seguras sin colisiones.\n\n4. Los movimientos peligrosos fueron\n   bloqueados correctamente.\n\n5. Limitaciones:\n   - FK simplificada\n   - no se usan sensores externos\n   - no existe planificacion dinamica\n\n6. El sistema es adecuado para\n   entornos controlados de laboratorio.\n'